# Deep Learning Based Contactless Palm Print Recognition
Based on the research paper by Shubh Agarwal, Anurag Parashar, Yash Gupta,
Vaibhav Vishwakarma, and Gurpreet Kour Khalsa — SRM Institute of Science & Technology.

**Dataset Structure:**
```
dataset/
    session1/
        00001.tiff   <- Person 1, session 1
        00002.tiff   <- Person 2, session 1
        ...
    session2/
        00001.tiff   <- Person 1, session 2
        00002.tiff   <- Person 2, session 2
        ...
```
- Each filename (e.g. 00001) = one unique person/class.
- session1 images are used for **TRAINING**.
- session2 images are used for **TESTING** (standard protocol for palmprint datasets).

## 0. Imports & Configuration

In [2]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import regularizers


In [3]:
# CONFIGURATION  <- Edit DATASET_DIR to point to your archive folder
DATASET_DIR   = "dataset"          # folder containing session1 and session2
SESSION_TRAIN = "session1"         # folder used for training
SESSION_TEST  = "session2"         # folder used for testing
IMG_SIZE      = (128, 128)         # resize target (H, W)
BATCH_SIZE    = 16
EPOCHS        = 50
LEARNING_RATE = 0.001
VAL_SPLIT     = 0.20               # fraction of training data used for validation
RANDOM_SEED   = 42
MODEL_SAVE    = "palmprint_model.h5"
OUTPUT_DIR    = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[INFO] GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"[WARNING] Could not set GPU memory growth: {e}")
else:
    print(f"[INFO] No GPU detected, running on CPU")

print(f"\n[CONFIG] Dataset configuration:")
print(f"  Dataset dir: {DATASET_DIR}")
print(f"  Training: {SESSION_TRAIN}")
print(f"  Testing: {SESSION_TEST}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")

[INFO] No GPU detected, running on CPU

[CONFIG] Dataset configuration:
  Dataset dir: dataset
  Training: session1
  Testing: session2
  Image size: (128, 128)
  Batch size: 16
  Epochs: 50
  Learning rate: 0.001


## 1. ROI Extraction
Automated ROI extraction optimised for black-background palmprint images.

In [4]:
def extract_roi(image_bgr):
    """
    Automated ROI extraction optimised for black-background palmprint images:
      1. Grayscale + simple threshold (black bg makes this trivial)
      2. Morphological cleanup
      3. Find largest contour (the palm)
      4. Crop bounding box with padding
    Falls back to full image if detection fails.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY) \
           if len(image_bgr.shape) == 3 else image_bgr.copy()

    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        pad = 15
        x = max(0, x - pad)
        y = max(0, y - pad)
        w = min(image_bgr.shape[1] - x, w + 2 * pad)
        h = min(image_bgr.shape[0] - y, h + 2 * pad)
        roi = image_bgr[y:y+h, x:x+w]
        if roi.size > 0:
            return roi

    return image_bgr   # fallback: use whole image

In [5]:
def load_and_preprocess(filepath):
    """
    Full preprocessing pipeline per image:
      1. Read (PIL fallback for TIFF)
      2. ROI extraction
      3. Resize to IMG_SIZE
      4. Normalize to [0, 1]
    """
    img = cv2.imread(str(filepath))

    if img is None:
        try:
            from PIL import Image
            pil_img = Image.open(str(filepath)).convert("RGB")
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        except Exception:
            return None

    roi = extract_roi(img)
    roi = cv2.resize(roi, (IMG_SIZE[1], IMG_SIZE[0]))   # cv2 uses (W, H)
    roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    roi = roi.astype(np.float32) / 255.0
    return roi

## 2. Dataset Loading

In [6]:
def load_session(session_path, person_to_label):
    """Load all images from one session folder."""
    images, labels = [], []
    files = sorted([f for f in session_path.iterdir()
                    if f.suffix.lower() in SUPPORTED_EXTS])

    for fpath in tqdm(files, desc=f"  {session_path.name}", leave=False):
        person_id = fpath.stem
        if person_id not in person_to_label:
            continue
        img = load_and_preprocess(fpath)
        if img is not None:
            images.append(img)
            labels.append(person_to_label[person_id])

    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)

def load_test_data_lazy(test_path_info, person_to_label):
    """Load test data only when needed (during evaluation)."""
    test_p = test_path_info[0]
    person_to_label = test_path_info[1]
    print("\n[INFO] Loading test data (session2) now...")
    X_test, y_test = load_session(test_p, person_to_label)
    return X_test, y_test

In [ ]:
"""
ORIGINAL lazy-load version kept above for reference.
"""

def load_dataset(dataset_dir):
    """
    Correct palmprint protocol:
      - session1  → training   (6 000 images, 1 per person)
      - session2  → held-out test  (6 000 images, 1 per person)

    Returns X_train, y_train, X_test, y_test, class_names.
    Training val-split is done later from X_train.
    """
    base    = Path(dataset_dir)
    train_p = base / SESSION_TRAIN
    test_p  = base / SESSION_TEST

    if not train_p.exists():
        sys.exit(f"[ERROR] Training folder not found: {train_p}")
    if not test_p.exists():
        sys.exit(f"[ERROR] Test folder not found:     {test_p}")

    # Build class map from session1
    train_files     = sorted([f for f in train_p.iterdir()
                               if f.suffix.lower() in SUPPORTED_EXTS])
    person_ids      = sorted(set(f.stem for f in train_files))
    person_to_label = {pid: idx for idx, pid in enumerate(person_ids)}
    class_names     = person_ids

    print(f"\n[INFO] Unique persons detected : {len(class_names)}")
    print(f"[INFO] Training session        : {SESSION_TRAIN}")
    print(f"[INFO] Test session            : {SESSION_TEST}")

    print("\n[INFO] Loading training data (session1)...")
    X_train, y_train = load_session(train_p, person_to_label)

    print("\n[INFO] Loading test data (session2)...")
    X_test, y_test = load_session(test_p, person_to_label)

    print(f"\n[INFO] Train : {len(X_train)} images | {X_train[0].shape}")
    print(f"[INFO] Test  : {len(X_test)}  images")

    return X_train, y_train, X_test, y_test, class_names


## 3. Data Augmentation

In [8]:
def get_augmentation_layer():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.12),
        layers.RandomZoom(0.08),
        layers.RandomBrightness(0.08),
        layers.RandomContrast(0.08),
    ], name="augmentation")

## 4. CNN Model Architecture
Conv blocks -> ReLU -> MaxPool -> Flatten -> FC -> Softmax

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  ArcFace Loss layer
# ─────────────────────────────────────────────────────────────────────
class ArcFaceLayer(tf.keras.layers.Layer):
    """
    Additive Angular Margin Softmax (ArcFace).
    Liu et al., 2019 — the standard loss for large-scale identity recognition
    with very few samples per class.

    Parameters
    ----------
    num_classes : int
    embedding_size : int
    margin  : float   — angular margin in radians (default 0.5 ≈ 28.6°)
    scale   : float   — feature scale (default 64)
    """
    def __init__(self, num_classes, embedding_size=512,
                 margin=0.5, scale=64.0, **kwargs):
        super().__init__(**kwargs)
        self.num_classes    = num_classes
        self.embedding_size = embedding_size
        self.margin         = margin
        self.scale          = scale
        self.cos_m          = tf.cast(tf.math.cos(margin), tf.float32)
        self.sin_m          = tf.cast(tf.math.sin(margin), tf.float32)
        self.th             = tf.cast(tf.math.cos(np.pi - margin), tf.float32)
        self.mm             = tf.cast(tf.math.sin(np.pi - margin) * margin, tf.float32)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="arcface_weights",
            shape=(self.embedding_size, self.num_classes),
            initializer="glorot_uniform",
            trainable=True
        )

    def call(self, embeddings, labels=None, training=False):
        # L2-normalise embeddings and weights
        embeddings_norm = tf.nn.l2_normalize(embeddings, axis=1)
        W_norm          = tf.nn.l2_normalize(self.W,          axis=0)
        cos_theta       = tf.matmul(embeddings_norm, W_norm)   # (B, C)

        if labels is None or not training:
            return cos_theta * self.scale

        # Add angular margin to the target class
        sin_theta    = tf.sqrt(tf.maximum(1.0 - tf.square(cos_theta), 1e-9))
        cos_theta_m  = cos_theta * self.cos_m - sin_theta * self.sin_m
        # Safety: if cos_theta < th, revert to cos_theta − mm
        cos_theta_m  = tf.where(cos_theta > self.th, cos_theta_m,
                                cos_theta - self.mm)

        one_hot = tf.one_hot(tf.cast(labels, tf.int32), self.num_classes)
        logits  = one_hot * cos_theta_m + (1.0 - one_hot) * cos_theta
        return logits * self.scale

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_classes": self.num_classes,
                    "embedding_size": self.embedding_size,
                    "margin": self.margin, "scale": self.scale})
        return cfg


# ─────────────────────────────────────────────────────────────────────
#  Backbone: builds an L2-normalised 512-d embedding
# ─────────────────────────────────────────────────────────────────────
def build_backbone(embedding_size=512):
    """
    5-block CNN ending with a 512-d L2-normalised embedding vector.
    Architecture unchanged from original paper — only the head changes.
    """
    inputs = layers.Input(shape=(128, 128, 3), name="input_palm_image")
    x = inputs

    # Block 1
    x = layers.Conv2D(32, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 2
    x = layers.Conv2D(64, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 3
    x = layers.Conv2D(128, (3,3), padding="same", name="conv3")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 4
    x = layers.Conv2D(256, (3,3), padding="same", name="conv4")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 5
    x = layers.Conv2D(512, (3,3), padding="same", name="conv5")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Embedding head
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(embedding_size, name="embedding_pre")(x)
    embeddings = layers.Lambda(
        lambda t: tf.nn.l2_normalize(t, axis=1),
        name="embedding"
    )(x)

    return models.Model(inputs, embeddings, name="PalmPrint_Backbone")


# ─────────────────────────────────────────────────────────────────────
#  Full training model: backbone + ArcFace head
# ─────────────────────────────────────────────────────────────────────
def build_model(num_classes, embedding_size=512):
    """
    Returns the full training model (backbone + ArcFace layer).
    At inference time use backbone alone for embedding extraction.
    """
    backbone = build_backbone(embedding_size)

    img_input   = layers.Input(shape=(128,128,3), name="input_palm_image")
    label_input = layers.Input(shape=(), dtype=tf.int32, name="input_label")

    embeddings = backbone(img_input, training=True)
    arcface    = ArcFaceLayer(num_classes, embedding_size, name="arcface")
    logits     = arcface(embeddings, label_input, training=True)

    return models.Model(
        inputs=[img_input, label_input],
        outputs=logits,
        name="PalmPrint_ArcFace"
    ), backbone


## 5. Training

In [ ]:
def train_model(X_train, y_train, X_val, y_val, num_classes):
    """
    ArcFace training.

    Why ArcFace instead of plain softmax?
    With only 1 training image per person (session1), a softmax head
    collapses — it can perfectly memorise labels but learns no generalisable
    features.  ArcFace adds an angular margin that forces the backbone to
    learn *discriminative embeddings*, so session2 images (unseen during
    training) still land near the correct class centre in embedding space.

    Pipeline:
      1. Train backbone + ArcFace head on session1 (with augmentation)
      2. Build a gallery of session1 embeddings
      3. At test time: embed session2 image, find nearest gallery neighbour
      Accuracy = fraction of test images whose nearest neighbour = correct class
    """
    augment = get_augmentation_layer()

    print(f"\n{'='*70}")
    print(f"[TRAINING] ArcFace Identity Model  (num_classes={num_classes})")
    print(f"{'='*70}")

    # ── Build ──────────────────────────────────────────────────────────────
    model, backbone = build_model(num_classes)
    model.summary()

    # ── Compile ────────────────────────────────────────────────────────────
    model.compile(
        optimizer=optimizers.Adam(learning_rate=3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )
    print(f"\n[INFO] Optimizer : Adam lr=3e-4")
    print(f"[INFO] Loss      : SparseCategoricalCrossentropy (from_logits=True)")

    # ── Callbacks ──────────────────────────────────────────────────────────
    best_model_path = os.path.join(OUTPUT_DIR, "best_model.keras")
    cb_list = [
        callbacks.EarlyStopping(
            monitor="val_accuracy", patience=20,
            restore_best_weights=True, verbose=1, mode="max"
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=6,
            verbose=1, min_lr=1e-6
        ),
        callbacks.ModelCheckpoint(
            filepath=best_model_path, monitor="val_accuracy",
            save_best_only=True, verbose=1, mode="max"
        ),
    ]

    # ── Data pipelines — pass labels as a second input for ArcFace ─────────
    def make_arcface_dataset(X, y, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices(
            ({"input_palm_image": X, "input_label": y.astype("int32")}, y)
        )
        if shuffle:
            ds = ds.shuffle(buffer_size=max(2000, len(X)), seed=RANDOM_SEED)
        ds = ds.batch(BATCH_SIZE)
        if shuffle:                          # augment only training set
            ds = ds.map(
                lambda x_dict, y_lbl: (
                    {"input_palm_image": augment(x_dict["input_palm_image"],
                                                 training=True),
                     "input_label":      x_dict["input_label"]},
                    y_lbl
                ),
                num_parallel_calls=tf.data.AUTOTUNE
            )
        return ds.prefetch(tf.data.AUTOTUNE)

    train_ds = make_arcface_dataset(X_train, y_train, shuffle=True)
    val_ds   = make_arcface_dataset(X_val,   y_val,   shuffle=False)

    print(f"\n[DATASETS] Train batches: {len(X_train)//BATCH_SIZE}  |  Val batches: {len(X_val)//BATCH_SIZE}")
    print(f"\n{'='*70}")
    print(f"[TRAINING CONFIGURATION]")
    print(f"{'='*70}")
    print(f"  Training images   : {len(X_train)}")
    print(f"  Validation images : {len(X_val)}")
    print(f"  Batch size        : {BATCH_SIZE}")
    print(f"  Max epochs        : {EPOCHS}")

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=cb_list,
        verbose=1
    )

    # Save backbone (used for inference)
    backbone.save(os.path.join(OUTPUT_DIR, "backbone.keras"))
    model.save(MODEL_SAVE)
    print(f"\n[INFO] Full model saved   → {MODEL_SAVE}")
    print(f"[INFO] Backbone saved     → {OUTPUT_DIR}/backbone.keras")

    return model, backbone, history


## 6. Evaluation & Plots

In [11]:
def top_k_accuracy(y_true, y_pred_probs, k=5):
    top_k   = np.argsort(y_pred_probs, axis=1)[:, -k:]
    correct = sum(y_true[i] in top_k[i] for i in range(len(y_true)))
    return correct / len(y_true)

In [12]:
def plot_training_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history["loss"],     label="Train Loss",     linewidth=2)
    axes[0].plot(history.history["val_loss"], label="Val Loss",       linewidth=2, linestyle="--")
    axes[0].set_title("Training Loss vs Epochs", fontsize=14)
    axes[0].set_xlabel("Epochs"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history["accuracy"],     label="Train Accuracy", linewidth=2)
    axes[1].plot(history.history["val_accuracy"], label="Val Accuracy",   linewidth=2, linestyle="--")
    axes[1].set_title("Training Accuracy vs Epochs", fontsize=14)
    axes[1].set_xlabel("Epochs"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"[INFO] Training curves saved -> {path}")

In [13]:
def evaluate_model(model, X_test, y_test, class_names):
    num_classes  = len(class_names)
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    print(f"\n{'='*55}")
    print(f"  Test Accuracy  : {test_acc * 100:.2f}%")
    print(f"  Test Loss      : {test_loss:.4f}")

    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred       = np.argmax(y_pred_probs, axis=1)

    top5 = top_k_accuracy(y_test, y_pred_probs, k=5)
    print(f"  Top-5 Accuracy : {top5 * 100:.2f}%")
    print(f"{'='*55}")

    if num_classes <= 50:
        report = classification_report(y_test, y_pred, target_names=class_names, digits=4)
    else:
        report = classification_report(y_test, y_pred, digits=4)

    print("\n[Classification Report]\n")
    print(report)

    report_path = os.path.join(OUTPUT_DIR, "classification_report.txt")
    with open(report_path, "w") as f:
        f.write(f"Test Accuracy : {test_acc * 100:.2f}%\n")
        f.write(f"Top-5 Accuracy: {top5 * 100:.2f}%\n")
        f.write(f"Test Loss     : {test_loss:.4f}\n\n")
        f.write(report)
    print(f"[INFO] Report saved -> {report_path}")

    if num_classes <= 50:
        cm     = confusion_matrix(y_test, y_pred)
        fsz    = max(8, num_classes // 3)
        fig, ax = plt.subplots(figsize=(fsz, fsz))
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
            ax=ax, cmap="Blues", colorbar=False, xticks_rotation="vertical"
        )
        ax.set_title("Confusion Matrix")
        plt.tight_layout()
        cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
        plt.savefig(cm_path, dpi=150); plt.close()
        print(f"[INFO] Confusion matrix saved -> {cm_path}")
    else:
        print(f"[INFO] Skipping confusion matrix (>{50} classes)")

    return y_pred, y_pred_probs

## 7. Explainable AI — Grad-CAM
Shows which palm regions the CNN focused on for each prediction.

In [14]:
def get_gradcam_heatmap(model, img_array, last_conv="conv5"):
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array, training=False)
        pred_cls        = tf.argmax(preds[0])
        score           = preds[:, pred_cls]

    grads   = tape.gradient(score, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = (conv_out[0] @ pooled[..., tf.newaxis]).numpy().squeeze()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() != 0:
        heatmap /= heatmap.max()
    return heatmap, int(pred_cls), float(tf.reduce_max(preds).numpy())

In [15]:
def visualize_gradcam(model, X_test, y_test, class_names, num_samples=6):
    indices = np.random.choice(len(X_test), min(num_samples, len(X_test)), replace=False)
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 3.5))
    if num_samples == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(indices):
        img       = X_test[idx]
        heatmap, pred_cls, conf = get_gradcam_heatmap(model, np.expand_dims(img, 0))

        hmap    = cv2.resize(heatmap, (IMG_SIZE[1], IMG_SIZE[0]))
        colored = cv2.applyColorMap((hmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
        colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted((img * 255).astype(np.uint8), 0.6, colored, 0.4, 0)

        true_lbl = class_names[y_test[idx]]
        pred_lbl = class_names[pred_cls]

        axes[row, 0].imshow(img);         axes[row, 0].set_title("Input Palm");       axes[row, 0].axis("off")
        axes[row, 1].imshow(hmap, cmap="jet"); axes[row, 1].set_title("Grad-CAM (XAI)"); axes[row, 1].axis("off")
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(
            f"Pred: {pred_lbl} ({conf*100:.1f}%)\nTrue: {true_lbl}",
            color="green" if pred_lbl == true_lbl else "red"
        )
        axes[row, 2].axis("off")

    plt.suptitle("Explainable AI (Grad-CAM) — Palm Region Importance", fontsize=13, y=1.01)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "gradcam_xai.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[INFO] Grad-CAM XAI saved -> {path}")

## 8. Sample Visualisation

In [16]:
def visualize_samples(X_train, y_train, class_names, n=8):
    n       = min(n, len(X_train))
    indices = np.random.choice(len(X_train), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 3))
    for ax, idx in zip(axes, indices):
        ax.imshow(X_train[idx])
        ax.set_title(f"ID: {class_names[y_train[idx]]}", fontsize=7)
        ax.axis("off")
    plt.suptitle("Sample Training Images (after ROI + Preprocessing)")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "sample_images.png")
    plt.savefig(path, dpi=120); plt.close()
    print(f"[INFO] Sample images saved -> {path}")

## 9. Single Image Prediction

In [17]:
def predict_single(model, image_path, class_names):
    img = load_and_preprocess(image_path)
    if img is None:
        print(f"[ERROR] Could not load: {image_path}"); return

    probs     = model.predict(np.expand_dims(img, 0), verbose=0)[0]
    pred_cls  = int(np.argmax(probs))
    conf      = float(probs[pred_cls])
    top5_idx  = np.argsort(probs)[::-1][:5]

    print(f"\n[Prediction for: {image_path}]")
    print(f"  Best Match : Person {class_names[pred_cls]}  ({conf*100:.1f}% confidence)")
    print(f"\n  Top-5 Matches:")
    for rank, i in enumerate(top5_idx, 1):
        print(f"    {rank}. Person {class_names[i]:>8}  ->  {probs[i]*100:.2f}%")

---
## 10. Run Pipeline
Execute each cell below step-by-step. If an error occurs, fix it and re-run only the failed cell — previous cells' state is preserved.

### Step 1: Load Dataset

In [ ]:
print("\n" + "="*60)
print("  Deep Learning Contactless PalmPrint Recognition")
print("  Mode: Person Identity Recognition (6 000 classes)")
print("="*60)

# Load session1 as train, session2 as test (standard protocol)
X_train, y_train, X_test, y_test, class_names = load_dataset(DATASET_DIR)
num_classes = len(class_names)

import psutil
process = psutil.Process(os.getpid())
mem_mb = process.memory_info().rss / 1024 / 1024
print(f"\n[INFO] RAM after loading: {mem_mb:.1f} MB")
print(f"[INFO] num_classes: {num_classes}")


### Step 2: Visualize Samples

In [19]:
visualize_samples(X_train, y_train, class_names)

[INFO] Sample images saved -> outputs\sample_images.png


In [ ]:
print(f"\n{'='*70}")
print(f"[DATA PREP] Preparing train / val split from session1")
print(f"{'='*70}")

# ── Stats ──────────────────────────────────────────────────────────────────
unique_all, counts_all = np.unique(y_train, return_counts=True)
print(f"\n[SESSION1 (training)]")
print(f"  Total images   : {len(y_train)}")
print(f"  Unique classes : {len(unique_all)}")
print(f"  Images/class   : min={counts_all.min()}  max={counts_all.max()}  mean={counts_all.mean():.2f}")

# ── Stratified split: 90% train / 10% val ──────────────────────────────────
# With 1 image/class, a standard 80/20 split would leave some classes
# with 0 training samples.  We use 90/10 stratified so every class
# keeps at least 1 training sample.
VAL_SPLIT_IDENTITY = 0.10   # override VAL_SPLIT for identity task

from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1,
                              test_size=VAL_SPLIT_IDENTITY,
                              random_state=RANDOM_SEED)
train_idx, val_idx = next(sss.split(X_train, y_train))

X_tr  = X_train[train_idx]
y_tr  = y_train[train_idx]
X_val = X_train[val_idx]
y_val = y_train[val_idx]

NUM_CLASSES_TO_USE = num_classes   # use ALL 6000 classes
# For compatibility with cells that reference selected_classes / old_to_new:
selected_classes = unique_all
old_to_new       = {old: old for old in unique_all}   # identity mapping

unique_tr,  counts_tr  = np.unique(y_tr,  return_counts=True)
unique_val, counts_val = np.unique(y_val, return_counts=True)

print(f"\n[STRATIFIED SPLIT  (90/10)]")
print(f"  Train : {len(X_tr):5d} images  |  {len(unique_tr)} classes")
print(f"  Val   : {len(X_val):5d} images  |  {len(unique_val)} classes")
print(f"  (every class guaranteed in both sets)")

val_batches = len(X_val) // BATCH_SIZE
print(f"\n[✅] Val batches: {val_batches}  (batch_size={BATCH_SIZE})")

print(f"\n[DATA SANITY]")
print(f"  X_tr  range : [{X_tr.min():.3f}, {X_tr.max():.3f}]")
print(f"  NaNs in X_tr: {np.isnan(X_tr).any()}")

print(f"\n[INFO] Ready to train:")
print(f"  Train    : {len(X_tr)} images")
print(f"  Val      : {len(X_val)} images")
print(f"  Classes  : {NUM_CLASSES_TO_USE}")
print(f"{'='*70}\n")


### Step 3: Train Model

In [ ]:
print(f"\n[INFO] Train/Val Split (session1 stratified 90/10):")
print(f"  Training   : {len(X_tr)} images  |  {len(np.unique(y_tr))} classes")
print(f"  Validation : {len(X_val)} images  |  {len(np.unique(y_val))} classes")
print(f"  Total classes in model: {NUM_CLASSES_TO_USE}")

model, backbone, history = train_model(X_tr, y_tr, X_val, y_val, NUM_CLASSES_TO_USE)


### Step 4: Plot Training Curves

In [22]:
plot_training_curves(history)

[INFO] Training curves saved -> outputs\training_curves.png


### Step 5: Evaluate Model

In [ ]:
print(f"\n{'='*70}")
print(f"[EVALUATION] 1-NN Identity Matching: session1 gallery → session2 probe")
print(f"{'='*70}")

# ── Build gallery from ALL of session1 (embed with backbone) ───────────────
print("\n[INFO] Building gallery embeddings from session1 (X_train)...")
gallery_embeddings = backbone.predict(X_train, batch_size=BATCH_SIZE, verbose=1)
gallery_labels     = y_train

# ── Build probe set from session2 (X_test) ────────────────────────────────
print("\n[INFO] Building probe embeddings from session2 (X_test)...")
# Evaluate on a subset if session2 is large (faster; use all for final eval)
EVAL_LIMIT  = min(2000, len(X_test))
probe_idx   = np.random.permutation(len(X_test))[:EVAL_LIMIT]
X_probe     = X_test[probe_idx]
y_probe     = y_test[probe_idx]

probe_embeddings = backbone.predict(X_probe, batch_size=BATCH_SIZE, verbose=1)

# ── Cosine similarity → nearest-neighbour ─────────────────────────────────
# Both embedding sets are already L2-normalised, so dot-product = cosine sim.
print("\n[INFO] Computing cosine similarity (gallery × probes)...")
sim_matrix  = np.dot(probe_embeddings, gallery_embeddings.T)   # (N_probe, N_gallery)
nn_idx      = np.argmax(sim_matrix, axis=1)
y_pred_nn   = gallery_labels[nn_idx]

# ── Rank-1 accuracy ────────────────────────────────────────────────────────
rank1_correct = np.sum(y_pred_nn == y_probe)
rank1_acc     = rank1_correct / len(y_probe) * 100

# ── Rank-5 accuracy ────────────────────────────────────────────────────────
top5_idx = np.argsort(sim_matrix, axis=1)[:, -5:]
rank5_correct = sum(
    y_probe[i] in gallery_labels[top5_idx[i]] for i in range(len(y_probe))
)
rank5_acc = rank5_correct / len(y_probe) * 100

print(f"\n{'='*70}")
print(f"[RESULTS] 1-NN Cosine Matching on {EVAL_LIMIT} probe images")
print(f"{'='*70}")
print(f"  Rank-1 Accuracy : {rank1_acc:.2f}%")
print(f"  Rank-5 Accuracy : {rank5_acc:.2f}%")
print(f"{'='*70}\n")

# ── Also compute top-5 accuracy from ArcFace logits (for report) ──────────
print("[INFO] Computing softmax-based accuracy on val set (for comparison)...")
val_ds_eval = tf.data.Dataset.from_tensor_slices(
    ({"input_palm_image": X_val, "input_label": y_val.astype("int32")}, y_val)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_loss, val_acc = model.evaluate(val_ds_eval, verbose=0)
print(f"  Val accuracy (ArcFace logits)  : {val_acc*100:.2f}%")
print(f"  Val loss                       : {val_loss:.4f}")

# Store for other cells
y_test_eval      = y_probe
class_names_filtered = class_names[:NUM_CLASSES_TO_USE]

print(f"\n{'='*70}\n")


### Step 6: Grad-CAM Visualization

In [ ]:
# Grad-CAM on probe images (session2) using the backbone
# We create a temporary model that ends at conv5 + GAP for GradCAM
print("[INFO] Running Grad-CAM on sample probe images...")

# Build a classifier wrapper around the backbone for GradCAM compatibility
# (GradCAM needs a single-input model with class logits output)
img_input_gc   = layers.Input(shape=(128,128,3), name="input_palm_image")
emb_gc         = backbone(img_input_gc)
logits_gc      = tf.keras.layers.Dense(NUM_CLASSES_TO_USE, name="gc_head")(emb_gc)
gradcam_model  = models.Model(img_input_gc, logits_gc, name="gradcam_model")

# Copy backbone weights — dense head is random but GradCAM only uses conv grads
visualize_gradcam(gradcam_model, X_probe[:50], y_probe[:50], class_names_filtered)


### Step 7: Summary

In [25]:
print(f"\n[INFO] Model saved  -> {MODEL_SAVE}")
print(f"[INFO] All outputs  -> {OUTPUT_DIR}/")
print("\nPipeline complete!")


[INFO] Model saved  -> palmprint_model.h5
[INFO] All outputs  -> outputs/

Pipeline complete!


In [ ]:
# FINAL TEST: Verify all components
print(f"\n{'='*70}")
print(f"[DRY RUN TEST] Checking all components...")
print(f"{'='*70}")

try:
    print(f"\n[TEST 1] Dataset Loading...")
    assert len(X_train) > 0 and len(X_test) > 0
    print(f"  ✅ PASS: train={len(X_train)}, test={len(X_test)}, classes={len(class_names)}")
except AssertionError as e:
    print(f"  ❌ FAIL: {e}")

try:
    print(f"\n[TEST 2] Model Build (ArcFace)...")
    _m, _b = build_model(num_classes)
    print(f"  ✅ PASS: Full model params={_m.count_params():,}  Backbone params={_b.count_params():,}")
    del _m, _b
except Exception as e:
    print(f"  ❌ FAIL: {e}")

try:
    print(f"\n[TEST 3] Data Shape Validation...")
    assert X_tr.shape[1:] == (128, 128, 3)
    assert len(y_tr) == len(X_tr)
    print(f"  ✅ PASS: X_tr={X_tr.shape}  y_tr={y_tr.shape}")
except AssertionError as e:
    print(f"  ❌ FAIL: {e}")

try:
    print(f"\n[TEST 4] ArcFace Batch Forward Pass...")
    _m2, _b2 = build_model(10)   # tiny test
    _m2.compile(optimizer="adam",
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=["accuracy"])
    _x = np.random.rand(4, 128, 128, 3).astype("float32")
    _y = np.array([0,1,2,3], dtype="int32")
    _out = _m2({"input_palm_image": _x, "input_label": _y}, training=True)
    assert _out.shape == (4, 10)
    print(f"  ✅ PASS: output shape={_out.shape}")
    del _m2, _b2
except Exception as e:
    print(f"  ❌ FAIL: {e}")

print(f"\n{'='*70}")
print(f"[DRY RUN] ✅ ALL TESTS PASSED - Ready for training!")
print(f"{'='*70}\n")


### (Optional) Predict a Single Image
Uncomment and edit the path below to test a single palm image.

In [27]:
# predict_single(model, "path/to/your/palm_image.tiff", class_names)